# 🧪 NutriTrack — Complete Test Suite

Runs all test scripts directly by importing functions from the `app/tests` directory.
- Output metrics are gathered into specific DataFrames.
- Raw data responses (p1-p4, q1-q6) are saved to `app/data/tests` for deep inspection.
- API Server is briefly spawned inside the notebook to run API endpoint tests.

## 0. Setup & Imports

In [1]:
import sys, os, time, json
import pandas as pd
from IPython.display import display

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, "config", ".env"))

import subprocess
import requests
import time
import tests.test_api as api_tests

FAST_FOOD_IMG = os.path.join(project_root, "..", "data", "images", "food", "fast_food.jpg")
COM_TAM_IMG   = os.path.join(project_root, "..", "data", "images", "food", "com_tam.jpg")

TEST_OUTPUT_DIR = os.path.join(project_root, "data", "tests")
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

def save_raw_output(filename: str, data: dict|list):
    if not data: return
    path = os.path.join(TEST_OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {filename}")

print(f"✅ Setup complete. Outputs will be saved to: {TEST_OUTPUT_DIR}")

✅ Setup complete. Outputs will be saved to: d:\Project\Code\nutritrack-documentation\app\data\tests


## 1. USDA Client Tests

In [2]:
from third_apis.USDA import USDAClient
import tests.test_usda_client as usda_tests

usda_client = USDAClient(api_key=os.getenv("USDA_API_KEY"))

2026-03-07 21:46:11 | INFO     | third_apis.USDA                | USDAClient initialized (api_key=***)


In [3]:
usda_results = usda_tests.run_all(usda_client)
print(f"\nUSDA Client tests passed: {sum(1 for r in usda_results if r)}/{len(usda_results)}")

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║      Starting test: _normalize_query       ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'rice (gạo)' → 'rice'
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'bò (beef)' → 'bo'
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'gà (chicken)' → 'ga'
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'cá (fish)' → 'ca'
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'crème brûlée (dessert)' → 'creme brulee'
2026-03-07 21:46:11 | INFO     | tests.test_usda_client         | ✅ PASS | 'pâté (duck)' → 'pate'

## 2. Qwen3VL Model Tests
Runs the 3 model inference methods directly against the test images and records metrics.

In [4]:
from models.QWEN3VL import Qwen3VL
import tests.test_qwen_client as qwen_tests

qwen = Qwen3VL()

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-07 21:45:13 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1


In [ ]:
qwen_results = qwen_tests.run_all(qwen)
for i, r in enumerate(qwen_results):
    # Extract method slug like 'converse', 'tools', etc.
    method_slug = r['method'].lower().replace(' ', '_')
    save_raw_output(f"q{i+1}_{method_slug}_{r['image']}.json", r.get("raw_output"))


In [ ]:
qwen_results = qwen_results # Already defined

print(f"\nQwen3VL tests passed: {sum(1 for r in qwen_results if r.get('success', False))}/{len(qwen_results)}")
print("(Note: Instructor tests are expected to fail with bedrock JSON schema)")

df_qwen = pd.DataFrame(qwen_results)
cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'tool_rounds', 'notes']
cols = [c for c in cols if c in df_qwen.columns]

display(df_qwen[cols].fillna(''))

## 3. Pipeline Tests
Testing the high-level orchestrated pipeline flows.

In [5]:
import tests.test_pipeline as pipeline_tests

try:
    if qwen:
        pass
except:
    from models.QWEN3VL import Qwen3VL
    import tests.test_qwen_client as qwen_tests

    qwen = Qwen3VL()

pipe_results = pipeline_tests.run_all(qwen, usda_client)
for i, r in enumerate(pipe_results):
    method_slug = r['method'].lower().replace(' ', '_')
    save_raw_output(f"p{i+1}_pipeline_{method_slug}_{r['image']}.json", r.get("raw_output"))

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-07 21:48:39 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1
                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║   Pipeline Test: method='tools' [steak]    ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-07 21:48:39 | INFO     | third_apis.USDA                | L1 RAM cache

In [6]:
pipe_results = pipe_results # Already defined
print(f"\nPipeline tests passed: {sum(1 for r in pipe_results if r.get('success', False))}/{len(pipe_results)}")

df_pipe = pd.DataFrame(pipe_results)
pipe_cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'token_input', 'price_input', 'token_output', 'price_output', 'notes']
pipe_cols = [c for c in pipe_cols if c in df_pipe.columns]

df_results = df_pipe[pipe_cols].fillna('') 

output_dir = '../data/tests'
os.makedirs(output_dir, exist_ok=True)
df_results.to_csv(os.path.join(output_dir, 'df_results.csv'), index=False)

display(df_results)


Pipeline tests passed: 4/4


,method,image,status,time_s,dishes,ingredients,bedrock_calls,usda_calls,usda_cache_hits,token_input,price_input,token_output,price_output,notes
0,tools,steak,✅ PASS,32.0,1,4,3,6,0,7698,0.0041,917,0.0024,
1,manual,steak,✅ PASS,20.2,1,6,1,5,1,1389,0.0007,849,0.0023,
2,tools,fast_food,✅ PASS,131.2,11,36,3,23,1,11372,0.0060,6463,0.0172,
3,manual,fast_food,✅ PASS,128.4,12,46,1,22,24,1517,0.0008,6373,0.0170,


## 4. API Endpoint Tests
Automatically spawn the Uvicorn server, hit the endpoints, and close the server.

In [ ]:
# Start fastapi server in background
print("Starting FASTAPI server in background...")
env = os.environ.copy()
server_process = subprocess.Popen(
    [sys.executable, "templates/api.py"], 
    cwd=project_root, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.PIPE,
    env=env
)

In [4]:
try:
    # Try hit healthcheck to confirm up
    host = "http://192.168.1.10:8000⁠/health⁠"
    resp = requests.get(host, timeout=5)
    print("Server is UP.")
    
    api_results = api_tests.run_all()
    print(f"\nAPI tests passed: {sum(1 for r in api_results if r.get('success'))}/{len(api_results)}")
    
except Exception as e:
    print(f"Failed during API tests: {e}")
finally:
    # Teardown server
    try:
        print("Terminating server...")
        server_process.terminate()
        server_process.wait()
        print("Server terminated.")
    except: pass

Failed during API tests: Failed to parse: http://192.168.1.10:8000⁠/health⁠
Terminating server...
